In [1]:
import pandas as pd
import anndata
import ccAFv2

###########################################
# 1 — Paths
###########################################
input_file = "D:/Halima's Data/Thesis_2/RCode/Cell_Cycle_prediction_with_scATAC_Seq/paper1/Dolly/BuettnerESCData/Buettner_mESC_benchmark_clean.csv"
output_file = "D:/Halima's Data/Thesis_2/RCode/Cell_Cycle_prediction_with_scATAC_Seq/paper1/Dolly/BuettnerESCData/predicted_by_ccAFv2_Buettner.csv"

###########################################
# 2 — Load normalized Buettner data
###########################################
df = pd.read_csv(input_file)

# Extract ground truth from Cell_ID
df["GroundTruth"] = df["Cell_ID"].apply(lambda x: x.split("_")[0])  # G1_cell1_count → G1

# Remove non-gene columns
gene_df = df.drop(columns=["Cell_ID", "GroundTruth"])

# Set cell IDs as index (AnnData obs names)
gene_df.index = df["Cell_ID"]

###########################################
# 3 — Create AnnData (cells × genes)
###########################################
adata = anndata.AnnData(X=gene_df.values)
adata.obs_names = gene_df.index          # Cell IDs
adata.var_names = gene_df.columns        # Genes

print("AnnData created:", adata)

###########################################
# 4 — Run ccAFv2 (mouse mode)
###########################################
predicted_labels, probabilities = ccAFv2.predict_labels(
    adata,
    species="mouse",
    gene_id="symbol"
)

adata.obs["Predicted"] = predicted_labels

###########################################
# 5 — Build final DataFrame
###########################################
result = pd.DataFrame({
    "CellID": adata.obs_names,
    "Predicted": adata.obs["Predicted"].values,
    "GroundTruth": df["GroundTruth"].values
})

###########################################
# 6 — Save output
###########################################
result.to_csv(output_file, index=False)

print("🔥 ccAFv2 predictions saved to:", output_file)


AnnData created: AnnData object with n_obs × n_vars = 288 × 38222
Running ccAFv2:
    Preparing data for classification...
    Marker genes present in this dataset: 760
    Missing marker genes in this dataset: 100
  Predicting cell cycle state probabilities...
9/9 [==============================] - 0s 2ms/step
  Choosing cell cycle state...
Done.
🔥 ccAFv2 predictions saved to: D:/Halima's Data/Thesis_2/RCode/Cell_Cycle_prediction_with_scATAC_Seq/paper1/Dolly/BuettnerESCData/predicted_by_ccAFv2_Buettner.csv
